<p style='text-align: center;'>

## <p style='text-align: center;'> **This is the Notebook for the CIFO Project**

### <p style='text-align: center;'> **Group: C37 Vestigial**




Add a way for the class to find whether with the presence of elitism the selection of indivs to new_population from population is even or odd so that when slecting the last indivs to the new population a decision is made on which offspring is passed to the new population given that if the transfer_size is odd only one of the offsprings from the crossover of two parents can be passed.

Also implement verbose
Also implement error handling such as num_triangles > 100
Also implement visualization of fitness over generations with different hyperparameters as is done in the labs and find a way to save these records to use for reference in the project report
Also implement Simulated Annealing/ Random Search as a way to compare between these and GAs
Also consider implementing elitism as a percentage given that you will change population size when testing different hyperparameters on your GAs
Also revise fitness function and rendering of the canvas (For instance the triangles should not be able to superimpose each other)

In [16]:
import numpy as np
import matplotlib.pyplot as plt
from random import randint, choices, sample, random
import cv2
from abc import ABC, abstractmethod
from copy import deepcopy

In [17]:
class Solution(ABC):
    def __init__(self, repr=None):
        if repr == None:
            repr = self.random_initial_representation()
        self.repr = repr

    def __repr__(self):
        return str(self.repr)
    
    @abstractmethod
    def fitness(self):
        pass

    @abstractmethod
    def random_initial_representation(self):
        pass

## <center> Section Containing the Algorithms to test for the Optimization Problem <center>

##### These Include:
- Hill Climbing
- Simulated Annealing
- Random Search

### <center> **Hill Climbing Algorithm** <center>

In [18]:
def hill_climbing(initial_solution, maximization=False, max_iter=9999, verbose=False):
    current = initial_solution
    improved = True
    iter = 1
    seqsols = [current]

    while improved:
        if verbose:
            print(f'Current Solution: {current.repr} ({current}) with fitness {current.fitness()}')
            
        improved = False

        bestneighb = current.get_bestneighbor()

        if maximization and bestneighb.fitness() >= current.fitness():
            current = deepcopy(bestneighb)
            improved = True
        elif not maximization and bestneighb.fitness() <= current.fitness():
            current = deepcopy(bestneighb)
            improved = True
            
        if improved:
            seqsols.append(current)

        iter += 1
        if iter == max_iter:
            break
    if verbose:
        print(f'Discarded best neighbor: {bestneighb.repr} ({bestneighb}) with fitness {bestneighb.fitness()}')
        print(f'Best visited: {current.repr} ({current}) with fitness {current.fitness()}')
        
    return current, seqsols

### <center> **Simulated Annealing Algorithm** <center>

Create generic Solution class as seen in Labs with methods fitness and random initial representation is repr is None

In [19]:
def simulated_annealing(
        initial_solution: Solution,
        C: float,
        L: int,
        H: float,
        maximization: bool = True,
        max_iter: int = 10,
        verbose: bool = False
):
    """
    Implementation of the Simulated Annealing optimization algorithm.
    Params:
        - initial_solution: Initial solution to the optimization problem
        - C (float): Control parameter (temperature)
        - L (int): Number of iterations with same C (iterations before temperature is adjusted)
        - H (float): Decreasing rate of C (temperature adjustment rate)
        - maximization (bool): Is this a maximization problem?
        - max_iter (int): Maximum number of iterations of the outer loop
        - verbose (bool): If True, prints progress details during execution. Defaults to False.
    """
    # Initialize solution if None then random initial representation
    current_solution = initial_solution
    current_fitness = current_solution.fitness()

    iter = 1
    history = []

    if verbose:
        print(f'Initial Solution {current_solution.repr} with fitness {current_fitness}')

    while iter <= max_iter:
        # Intial loop (for L times) 
        for _ in range(L):
            random_neighbor = current_solution.get_random_neighbor()
            neighbor_fitness = random_neighbor.fitness()

            if verbose:
                print(f'Random neighbor {random_neighbor} with fitness {neighbor_fitness}')

            if (
                (maximization and (neighbor_fitness >= current_fitness))
                or(not maximization and (neighbor_fitness <= current_fitness))
            ):
                current_solution = deepcopy(random_neighbor)
                current_fitness = current_solution.fitness()

                if verbose:
                    print(f'Neighbor is better, current solution is replaced')

            else:

                random_float = random()
                # Define probability that SA will accept a worse fitness solution
                p = np.exp(-abs(current_fitness - neighbor_fitness) / C)
                if verbose:
                    print(f'Probability of accepting worse event: {p}')
                # The event happens with probability P if the random number is lower than P 
                if random_float < p:
                    current_solution = deepcopy(random_neighbor)
                    current_fitness = current_solution.fitness()
                    if verbose:
                        print(f'Neighbor is worse and was accepted')
                else:
                    if verbose:
                        print(f'Neighbor is worse and was not accepted') 
            history.append(current_solution)

            if verbose:
                print(f'History was appended with {current_solution} and fitness {current_fitness}')

        # Inner look (L) ended, update C and iter
        C = C / H
        if verbose:
            print(f'Decreased C, new value: {C}')
        iter += 1
    if verbose:
        print(f'Best solution found: {current_solution.repr} with fitness {current_fitness}')

    return current_solution, history

### <center> Generic Implementation of the Solution Class <center>


- This defines the attributes of the solution which are mandatory such as width, height and number of triangles.
- Implementation of a Genetic Algorithm is implemented in order to find the best solution at replicating the target image.

In [ ]:
class VermeerSolution(Solution):
    def __init__(self, target_image, repr=None):
        """"Class with the attributes highlighted within the project description width, heigh and num_triangles"""
        self.target_image = target_image
        self.width = 300
        self.height = 400
        self.num_triangles = 100
        super().__init__(repr=repr)

    def random_initial_representation(self):
        """
        Generates small, localized triangles to prevent early blobbing.
        """
        random_repr = np.zeros((self.num_triangles, 9), dtype=int)

        for i in range(self.num_triangles):
            # 1. Pick a random "center" for the triangle
            cx = randint(0, self.width)
            cy = randint(0, self.height)

            # 2. Generate vertices within a small radius (40 pixels) of the center
            radius = 40
            x1 = np.clip(cx + randint(-radius, radius), 0, self.width)
            y1 = np.clip(cy + randint(-radius, radius), 0, self.height)
            x2 = np.clip(cx + randint(-radius, radius), 0, self.width)
            y2 = np.clip(cy + randint(-radius, radius), 0, self.height)
            x3 = np.clip(cx + randint(-radius, radius), 0, self.width)
            y3 = np.clip(cy + randint(-radius, radius), 0, self.height)

            # 3. Generate random RGB
            r, g, b = [randint(0, 255) for _ in range(3)]

            random_repr[i] = [x1, y1, x2, y2, x3, y3, r, g, b]
        
        return random_repr
    
    def render_canvas(self):
        """
        Converts 100-triangle array into a 300x400 pixel image matrix
        """
        # Create a blank canvas again with 300 width, 400 height and 3 color chanels (r,g,b)
        canvas = np.zeros((self.height, self.width, 3), dtype=np.uint8)

        for gene in self.repr:
            # Extract the 3 (x,y) vertices and reshape them for OpenCV
            # Format: [[x1, y1], [x2, y2], [x3, y3]]
            pts = np.array([
                [gene[0], gene[1]],
                [gene[2], gene[3]],
                [gene[4], gene[5]]
            ], np.int32)

            pts = pts.reshape((-1, 1, 2))

            # Extract RGB 
            color = (int(gene[6]), int(gene[7]), int(gene[8]))
            cv2.fillPoly(canvas, [pts], color)
        return canvas
    

    def fitness(self):
        """
        Calculates RMSE and applies a soft penalty for massive triangles.
        """
        generated_image = self.render_canvas()
        
        target_float = self.target_image.astype(np.float32)
        gen_float = generated_image.astype(np.float32)
        
        # 1. Base RMSE Calculation
        diff = target_float - gen_float
        sq_diff = np.square(diff)
        mse = np.mean(sq_diff)
        rmse = np.sqrt(mse)

        # 2. The Size/Overlap Soft Penalty
        # Extract all X and Y coordinates to calculate area using the Shoelace Formula
        x1, y1 = self.repr[:, 0], self.repr[:, 1]
        x2, y2 = self.repr[:, 2], self.repr[:, 3]
        x3, y3 = self.repr[:, 4], self.repr[:, 5]

        # Calculate area of all 100 triangles at once
        areas = 0.5 * np.abs(x1*(y2 - y3) + x2*(y3 - y1) + x3*(y1 - y2))

        # Define a maximum acceptable area (e.g., a 40x40 pixel square is 1600 area)
        max_allowed_area = 1500
        
        # Find triangles that exceed this area
        excess_area = np.maximum(0, areas - max_allowed_area)

        # Create a penalty proportional to the excess size
        # The weight (0.001) determines how strict the GA is about large triangles.
        # You can tune this! Higher = strictly small triangles. Lower = some big triangles allowed.
        size_penalty = np.sum(excess_area) * 0.00001 

        # The final fitness is the visual error PLUS the geometric penalty
        return rmse + size_penalty


In [21]:
gonçalo_path = 'C:\CIfO\Project\data\girl_pearl_earing.png'
original_image = cv2.imread(r'C:\CIfO\Project\data\girl_pearl_earing.png')
first_painting = VermeerSolution(target_image=original_image)
print(f'{first_painting.repr} has fitness {first_painting.fitness()}')

[[176 198 202 233 173 210  41  71  32]
 [222 317 259 329 208 347  56 183 139]
 [ 86  29  48   0  28   0 142 232 176]
 [243 381 272 339 300 400 251  34  38]
 [217 258 228 273 244 312  52 102 172]
 [111 241  55 278  91 266 137 110   5]
 [ 49  54  69  14  22  14 178  33  96]
 [262 207 277 161 300 205 100 238  14]
 [263  63 208  52 251  15  10 229  65]
 [258 165 300 126 294 161   4 239 152]
 [263 248 207 241 221 248  58  97  47]
 [ 34 400   7 345  62 350 148  55 140]
 [137 149 179 145 208 177 219 251 125]
 [100 270 128 304 173 239  78 237 185]
 [296 194 275 125 263 145 121 149  72]
 [125  41 150  53  76  35 104  64 214]
 [290 300 214 289 262 304 156   4 113]
 [171 146 202 161 157 135 109 100  83]
 [232 189 186 254 168 226 234 167  53]
 [ 30 105  20 123  21 125  95  88 107]
 [128 343  97 324 126 294 154 249 158]
 [ 67 208  98 251  46 238 222 145 170]
 [ 41  21  75  12  14  78 133 191  14]
 [137 136 140  91 165  87  94 177 241]
 [142   3 203  11 129  13 131 177 233]
 [ 29 344  42 301   0 312

<>:1: SyntaxWarning: invalid escape sequence '\C'
<>:1: SyntaxWarning: invalid escape sequence '\C'
C:\Users\gpape\AppData\Local\Temp\ipykernel_26440\2813924900.py:1: SyntaxWarning: invalid escape sequence '\C'
  gonçalo_path = 'C:\CIfO\Project\data\girl_pearl_earing.png'


Now to test out if the initial class functions in producing an image and comparing it to the original painting

In [22]:
"""gonçalo_path = 'C:\CIfO\Project\data\girl_pearl_earing.png'

original_vermeer = cv2.imread(r'C:\CIfO\Project\data\girl_pearl_earing.png')

my_first_painting = VermeerSolution(target_image=original_vermeer)

score = round(my_first_painting.fitness(), 2)

print(f'The RMSE Score of random painting is: {score}')"""



<>:1: SyntaxWarning: invalid escape sequence '\C'
<>:1: SyntaxWarning: invalid escape sequence '\C'
C:\Users\gpape\AppData\Local\Temp\ipykernel_26440\1249356740.py:1: SyntaxWarning: invalid escape sequence '\C'
  """gonçalo_path = 'C:\CIfO\Project\data\girl_pearl_earing.png'


"gonçalo_path = 'C:\\CIfO\\Project\\data\\girl_pearl_earing.png'\n\noriginal_vermeer = cv2.imread(r'C:\\CIfO\\Project\\data\\girl_pearl_earing.png')\n\nmy_first_painting = VermeerSolution(target_image=original_vermeer)\n\nscore = round(my_first_painting.fitness(), 2)\n\nprint(f'The RMSE Score of random painting is: {score}')"

Now I will create my GA class to perform optimization of my solution

**Notes**:
The probability of mutation should be 1 over the length of the representation (string)

### New Implementation of the GA with upgrades Mutation, Crossover and Elitism

In [23]:
class VermeerGA2:
    def __init__(self, target_image, pop_size=200, generations=1000, pc = 0.85, pm = 0.03, elitism_size=1, tournament_size=3):
        self.target_image = target_image
        self.pop_size = pop_size
        self.generations = generations
        self.pc = pc
        self.pm = pm
        self.elitism_size = elitism_size
        self.tournament_size = tournament_size
        
        # Initialize the population with random paintings
        self.population = [VermeerSolution(target_image=self.target_image) for _ in range(self.pop_size)]

    def tournament_selection(self):
        """Picks K random individuals, returns the one with the lowest RMSE."""
        tournament = sample(self.population, self.tournament_size)
        # Sort by the pre-calculated fitness_score attribute
        winner = sorted(tournament, key=lambda ind: ind.fitness_score)[0]
        return winner

    def crossover(self, parent1, parent2):
        """Randomly selects each triangle either from Parent1 or Parent2"""
        child = VermeerSolution(target_image=self.target_image)
        # Create a boolean mask of 0s and 1s
        mask = np.random.randint(0, 2, size=(child.num_triangles, 1))
        
        # if mask == 1, take p1´s gene if 0, take p2´s gene
        child.repr = np.where(mask, parent1.repr, parent2.repr)
        return child

    def mutate(self, child):
        """Upgraded Mutation: Independent tuning and Smart Color sampling."""
        for i in range(child.num_triangles):
            if random() < self.pm:
                mutation_type = random()

                if mutation_type < 0.40:
                    # 1. SHAPE MUTATION (40%): Only nudge coordinates
                    child.repr[i][0:6] += np.random.randint(-15, 16, 6)
                    child.repr[i][0::2] = np.clip(child.repr[i][0::2], 0, child.width - 1)
                    child.repr[i][1::2] = np.clip(child.repr[i][1::2], 0, child.height - 1)

                elif mutation_type < 0.70:
                    # 2. COLOR MUTATION (30%): Only nudge colors
                    child.repr[i][6:9] += np.random.randint(-20, 21, 3)
                    child.repr[i][6:9] = np.clip(child.repr[i][6:9], 0, 255)

                elif mutation_type < 0.90:
                    # 3. SMART COLOR SYNC (20%): Sample the target image!
                    # Find the centroid (middle) of the triangle
                    cx = int((child.repr[i][0] + child.repr[i][2] + child.repr[i][4]) / 3)
                    cy = int((child.repr[i][1] + child.repr[i][3] + child.repr[i][5]) / 3)
                    
                    # Ensure coordinates are safely within bounds
                    cx = np.clip(cx, 0, child.width - 1)
                    cy = np.clip(cy, 0, child.height - 1)
                    
                    # OpenCV images are (Y, X, Channels) and BGR format
                    b, g, r = self.target_image[cy, cx]
                    
                    # Update the triangle's genes with the exact target color
                    child.repr[i][6:9] = [r, g, b]

                elif mutation_type < 0.95:
                    # 4. Z-INDEX SWAP (5%)
                    swap_idx = randint(0, child.num_triangles - 1)
                    child.repr[[i, swap_idx]] = child.repr[[swap_idx, i]]
                    
                else:
                    # 5. FULL RESET (5%): Move a stuck triangle to a new random location
                    # Using the localized spawn to prevent giant blobs!
                    cx, cy = randint(0, child.width), randint(0, child.height)
                    rad = 30
                    child.repr[i] = [
                        np.clip(cx + randint(-rad, rad), 0, child.width), np.clip(cy + randint(-rad, rad), 0, child.height),
                        np.clip(cx + randint(-rad, rad), 0, child.width), np.clip(cy + randint(-rad, rad), 0, child.height),
                        np.clip(cx + randint(-rad, rad), 0, child.width), np.clip(cy + randint(-rad, rad), 0, child.height),
                        randint(0, 255), randint(0, 255), randint(0, 255)
                    ]
        return child

    def run(self):
        """The main evolutionary loop."""
        print(f"Starting Evolution for {self.generations} generations...")
        
        # Track history for plotting in final report
        history = []

        for gen in range(self.generations):
            # 1. Evaluate the entire population
            for ind in self.population:
                if not hasattr(ind, 'fitness_score'):
                # We save the score as an attribute so we don't recalculate it constantly
                    ind.fitness_score = ind.fitness()
            
            # 2. Sort population from best (lowest RMSE) to worst
            self.population.sort(key=lambda ind: ind.fitness_score)
            history.append(self.population[0].fitness_score)
            
            if gen % 50 == 0:
                print(f"Generation {gen} | Best RMSE: {self.population[0].fitness_score:.2f}")
            
            # 3. Create the next generation with elitism (Copy k size individuals directly to next generation)
            next_generation = [deepcopy(ind) for ind in self.population[:self.elitism_size]]
            
            # 4. Breed the rest of the new generation
            while len(next_generation) < self.pop_size:
                # Select parents 
                p1 = self.tournament_selection()
                # Crossover with probability Pc
                if random() < self.pc:
                    p2 = self.tournament_selection()
                    child = self.crossover(p1, p2)
                else:
                    child = deepcopy(p1) # Replication with probality 1 - Pc
                
                # Mutate
                child = self.mutate(child)
                next_generation.append(child)
                
            # 5. Replace the old population with the new one P := P''
            self.population = next_generation
            
        print(f"Evolution Complete! Final RMSE: {self.population[0].fitness_score:.2f}")
        # Return the absolute best painting and the historical data
        return self.population[0], history

In [24]:
if __name__ == "__main__":
    # 1. Load the target image
    target = cv2.imread(r'C:\CIfO\Project\data\girl_pearl_earing.png')
    if target is None:
        raise FileNotFoundError("OpenCV could not find the image! Check the file name and folder path.")
    
    # 2. Initialize the Genetic Algorithm
    # Start with a small population (e.g., 20) and low generations (e.g., 50) 
    ga = VermeerGA2(target_image=target, pop_size=150, generations=6000, pc=0.85, pm = 0.05, elitism_size=10, tournament_size=3)
    
    # 3. Run the optimization!
    best_painting, fitness_history = ga.run()
    
    # 4. Render and view the final result
    final_image = best_painting.render_canvas()
    cv2.imshow("Best Vermeer Approximation", final_image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    

Starting Evolution for 6000 generations...
Generation 0 | Best RMSE: 100.15
Generation 50 | Best RMSE: 59.16
Generation 100 | Best RMSE: 49.01
Generation 150 | Best RMSE: 44.66
Generation 200 | Best RMSE: 41.95
Generation 250 | Best RMSE: 40.76
Generation 300 | Best RMSE: 40.12
Generation 350 | Best RMSE: 40.06
Generation 400 | Best RMSE: 40.06
Generation 450 | Best RMSE: 39.91
Generation 500 | Best RMSE: 39.22
Generation 550 | Best RMSE: 38.53
Generation 600 | Best RMSE: 38.49
Generation 650 | Best RMSE: 38.49
Generation 700 | Best RMSE: 38.49
Generation 750 | Best RMSE: 38.49
Generation 800 | Best RMSE: 38.49
Generation 850 | Best RMSE: 38.38
Generation 900 | Best RMSE: 38.13
Generation 950 | Best RMSE: 38.07
Generation 1000 | Best RMSE: 38.03
Generation 1050 | Best RMSE: 37.99
Generation 1100 | Best RMSE: 37.94
Generation 1150 | Best RMSE: 37.88
Generation 1200 | Best RMSE: 37.80
Generation 1250 | Best RMSE: 37.65
Generation 1300 | Best RMSE: 37.21
Generation 1350 | Best RMSE: 36.92

New Vermeer GA: 3rd Try: Hyperparameters: pop_size = 200, generations=2000, elitism_size=10, tournament_size=3 <br>
**RMSE**: 46:66

New Vermmer GA: 2nd Try: Hyperparameters: pop_size=100, generations=5000, elitism_size=10, tournament_size=6 <br>
**RMSE** : 46.55

New Vermeer GA: 
1st Try:
Hyperparameters: pop_size=200, generations=2000, elitism_size=10, tournament_size=6 <br>
**RMSE** : 46.88

In [25]:
"""
Run #1 with new GA (VermeerGA2):
Pop size = 200
Nº Generations = 1000
pc = 0.85
pm = 0.03 
elitism_size=1 
tournament_size=3
Best RMSE: 34.17
--------------------------
Run # 2 with new GA (VermeerGA2):
Pop size = 200
Nº Generations = 2000
pc = 0.85
pm = 0.03 
elitism_size=1 
tournament_size=3
Best RMSE: 31.67
--------------------------
Run # 3 with new GA (VermeerGA2):
Pop size = 200
Nº Generations = 2000
pc = 0.85
pm = 0.03 
elitism_size=3
tournament_size=6
Best RMSE: 30.35

"""

'\nRun #1 with new GA (VermeerGA2):\nPop size = 200\nNº Generations = 1000\npc = 0.85\npm = 0.03 \nelitism_size=1 \ntournament_size=3\nBest RMSE: 34.17\n--------------------------\nRun # 2 with new GA (VermeerGA2):\nPop size = 200\nNº Generations = 2000\npc = 0.85\npm = 0.03 \nelitism_size=1 \ntournament_size=3\nBest RMSE: 31.67\n--------------------------\nRun # 3 with new GA (VermeerGA2):\nPop size = 200\nNº Generations = 2000\npc = 0.85\npm = 0.03 \nelitism_size=3\ntournament_size=6\nBest RMSE: 30.35\n\n'